# Data and baselines

This notebook covers the dataset and the **clean baselines**: how well the model answers *before* any poisoning.
Every later result is compared against these numbers.

It produces `results/baselines_<model>_n<N>.csv` per model (one row per question and setting) and a `.json` file with the generation settings.

In [1]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

from ragtrust.runs import load_selection
from ragtrust.data import load_hotpotqa, pick_fewshot_examples
from ragtrust.analysis import add_match
from ragtrust.experiment import SETTINGS, cached_run, run_setting
from ragtrust.llm import OllamaLLM
from ragtrust.poison import find_answer_passage

MODELS = ["llama3.1:8b", "qwen2.5:7b"]
TAG = "final"   # part of every result file name
SEED = 0
RESULTS = Path("../results")

## 1. The dataset

HotpotQA, validation split, *distractor* setting: each question comes with 10 Wikipedia paragraphs,
of which 2 are **gold** (they contain the supporting facts) and 8 are **distractors** (on-topic but not needed).

In [2]:
examples = load_selection()

ex = examples[0]
print(ex.question, "->", ex.answer, f"({ex.qtype})")
for p in ex.passages:
    print("GOLD" if p.is_gold else "    ", p.title, "|", p.text[:80])

Name the movie that was cowritten by Paul Rudd and the director of the 2011 British science fiction horror comedy film written and directed by Joe Cornish and starring John Boyega, Nick Frost, Jodie Whittaker and Luke Treadaway. -> Ant-Man (bridge)
     The Adventures of Tintin (film) | The Adventures of Tintin, known as The Adventures of Tintin: The Secret of the U
     You Instead | You Instead (alternate U.S. title Tonight You're Mine) is a 2011 British "Rock '
     Hello Carter | Hello Carter is a 2013 British comedy-drama film written and directed by Anthony
     Paul (film) | Paul is a 2011 science fiction road comedy film directed by Greg Mottola and wri
GOLD Attack the Block | Attack the Block is a 2011 British science fiction horror comedy film written an
     Consuming Passion | Consuming Passion is a 2008 British drama which features three interwoven storie
     Brothers of the Head | Brothers of the Head is a 2005 mockumentary featuring the story of Tom and Barry
GOLD Joe C

In [3]:
print("Question types:", Counter(ex.qtype for ex in examples))
yes_no = sum(ex.answer.lower() in ("yes", "no") for ex in examples)
print(f"Yes/no answers: {yes_no} of {len(examples)}")
print(f"Mean answer length: {sum(len(ex.answer.split()) for ex in examples) / len(examples):.1f} words")

Question types: Counter({'bridge': 800})
Yes/no answers: 0 of 800
Mean answer length: 2.3 words


## 2. Answer format: few-shot demonstrations

Three question and short answer pairs from the **train** split (never from the evaluation questions, to avoid leakage):
one bridge, one comparison, one yes/no. They teach the model to answer briefly, which Exact Match requires.

In [4]:
fewshot = pick_fewshot_examples(seed=SEED)
for q, a in fewshot:
    print(q, "->", a)

Where is the base of one of the three main British intelligence agencies? -> Cheltenham
 Hugo Ferreira and Glenn Gregory are both singers and what? -> songwriter
Are Randal Kleiser and Kyle Schickner of the same nationality?  -> yes


## 3. Baselines: three settings, same questions

| Setting | Context | Measures |
|---|---|---|
| `closed_book` | none | what the model knows from memory |
| `gold_only` | the 2 gold paragraphs | upper bound with perfect evidence |
| `distractor` | all 10 paragraphs | realistic, noisy evidence |

Same system prompt in all three, so only the context changes.
Every model is run on the same questions; saved results are loaded instead of re-run.

In [5]:
def baseline_path(model):
    return RESULTS / f"baselines_{model.replace(':', '-')}_{TAG}.csv"

frames = []
for model in MODELS:
    llm = OllamaLLM(model=model, seed=SEED)
    run = lambda: pd.concat([run_setting(llm, examples, s, fewshot) for s in SETTINGS])
    frames.append(cached_run(baseline_path(model), llm, run).assign(model=model))

df = add_match(pd.concat(frames))


closed_book:   0%|          | 0/800 [00:00<?, ?it/s]


closed_book:   0%|          | 1/800 [00:08<1:52:19,  8.43s/it]


closed_book:   0%|          | 3/800 [00:08<29:44,  2.24s/it]  


closed_book:   1%|          | 5/800 [00:08<15:29,  1.17s/it]


closed_book:   1%|          | 7/800 [00:09<09:21,  1.41it/s]


closed_book:   1%|          | 9/800 [00:09<06:14,  2.11it/s]

... (4,043 outputs left out)


distractor: 100%|█████████▉| 798/800 [09:49<00:01,  1.36it/s]


distractor: 100%|█████████▉| 799/800 [09:49<00:00,  1.44it/s]


distractor: 100%|██████████| 800/800 [09:50<00:00,  1.41it/s]


distractor: 100%|██████████| 800/800 [09:50<00:00,  1.35it/s]

## 4. Results

In [6]:
df.groupby(["model", "setting"])[["em", "f1", "match"]].mean().round(3)

em     f1  match
model       setting                         
llama3.1:8b closed_book  0.346  0.410  0.375
            distractor   0.586  0.705  0.672
            gold_only    0.684  0.808  0.768
qwen2.5:7b  closed_book  0.246  0.316  0.278
            distractor   0.565  0.679  0.632
            gold_only    0.660  0.789  0.750

## 5. Known and unknown questions

An answer **matches** if the normalized gold answer appears in it as whole words (`contains_answer`).
We use this instead of EM for decisions, because EM counts right-but-wordy answers ("They are both American.") as wrong.

A question is **known** if the closed-book answer matches.
On known questions the model can compare a document with its memory; on unknown ones it can only rely on the documents.
This split will be applied to every poisoning result later.

In [7]:
# Rows: closed-book wrong/right. Columns: gold-only wrong/right.
wide = df.pivot_table(index=["model", "id"], columns="setting", values="match", aggfunc="first").astype(int)
for model in MODELS:
    print(model)
    display(pd.crosstab(wide.loc[model, "closed_book"], wide.loc[model, "gold_only"], margins=True))

llama3.1:8b


gold_only,0,1,All
closed_book,,,
0,171,329,500
1,15,285,300
All,186,614,800


qwen2.5:7b


gold_only,0,1,All
closed_book,,,
0,180,398,578
1,20,202,222
All,200,600,800


In [8]:
wide["known"] = wide["closed_book"] == 1
wide.groupby(["model", "known"])[["gold_only", "distractor"]].agg(["mean", "count"]).round(3)

setting           gold_only       distractor      
                       mean count       mean count
model       known                                 
llama3.1:8b False     0.658   500      0.546   500
            True      0.950   300      0.883   300
qwen2.5:7b  False     0.689   578      0.578   578
            True      0.910   222      0.775   222